In [ ]:
import requests
import json
import pandas as pd

url = "http://192.168.74.211:30530/telecso/_doc"  # auto-generate _id
auth = ("elastic", "LVY6XizwysCTc82HPi8XX0uY")  # basic auth

df_product = pd.read_csv('../data/dim_product_elastic.csv')

for idx, row in df_product.iterrows():
    doc = {
        "code": row["PRODUCT_CODE"],
        "name": row["PRODUCT_DESC"],
        "name_autocomplete": row["PRODUCT_DESC"],
        "name_synonym": row["PRODUCT_DESC"]
    }
    response = requests.post(url, auth=auth, json=doc)
    print(f"{idx}: {response.status_code}")

In [1]:
# loading environment variables 
from dotenv import load_dotenv
load_dotenv(override=True)  # take environment variables

True

In [2]:
import pandas as pd

df_product = pd.read_csv('../data/dim_product_mapping_goa.csv')

In [3]:
from langchain_milvus import Milvus
from langchain_openai import OpenAIEmbeddings
import os

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

milvus_db = os.getenv("MILVUS_DB")
milvus_host = os.getenv("MILVUS_HOST")
milvus_port = os.getenv("MILVUS_PORT")
milvus_collection = os.getenv("MILVUS_COLLECTION")

vectorstore = Milvus(
    embedding_function=embedding_model,
    connection_args={
        "uri": f"http://{milvus_host}:{milvus_port}",
        "db_name": milvus_db
    },
    collection_name=milvus_collection,
)

In [ ]:
df_product.columns

In [4]:
df_product = df_product[~df_product['product_sku'].isna()]

In [5]:
df_product["embedding_input"] = df_product.apply(
    lambda x: (
        f"Product Name: {x['product_name_aam']}; "
        f"Description: {x['product_short_description']}; "
        f"Indication: {x['product_indication']}; "
        f"Composition: {x['product_composition']}; "
        f"Dosage: {x['product_dosis']}; "
        f"Side Effects: {x['product_side_effects']}"
    ),
    axis=1
)

In [7]:
from uuid import uuid4
from langchain_core.documents import Document

documents = []
uuids = []

for _, row in df_product.iterrows():
    document = Document(
        page_content=row["embedding_input"],
        metadata={
            "name": str(row["product_name_aam"]),
            "code": str(row["product_code_aam"]),
            "description": str(row["product_short_description"]),
            "indication": str(row["product_indication"]),
            "composition": str(row["product_composition"]),
            "dosage": str(row["product_dosis"]),
            "side_effects": str(row["product_side_effects"]),
        }
    )
    documents.append(document)
    uuids.append(str(uuid4()))

In [12]:
vectorstore.add_documents(documents=documents, ids=uuids)

['86e0bd53-a922-4d6f-9f67-d4840efcdf62',
 '642f5e43-9f1a-4c78-806c-da4ca0e8e093',
 '6a91d695-108e-4b50-8026-c2480b526b7f',
 'bd658e60-5e8e-4eec-a9de-3c72b18c00ce',
 '404159bd-5ef5-4a08-85c1-7304a27f3e98',
 'c209ac9b-35bd-44a1-9f06-e255d527ed0e',
 '1c1ea19d-0ea5-41fc-9e3a-765f9f11c881',
 'e672cc6b-9ac4-44f9-9349-5ed8dd2d2fc4',
 '7877498e-8641-4eb8-84d8-1a9d199fe262',
 'dfeb5491-1450-401e-b3f4-580517be2603',
 'e2e1ef0b-dfde-41f6-b1fa-1bf8601c2a24',
 '36544dff-00f9-430f-aa95-02751429e704',
 '74c561a8-b8c4-4390-b0c4-f2313812ebaa',
 'cc2ee4d5-7ed8-486b-82e1-6a3215652767',
 'ba4be650-db25-41b2-a0be-8fda155369a8',
 '54b5f0fb-ae32-485d-b051-36b684aabdb0',
 '0d486cf9-1f7c-4edf-a665-6db1be1789ac',
 '83420f0d-dff5-4903-aa08-fe925fd79950',
 '42def515-84cd-4871-a543-64adfb7bf23a',
 '4ddbc2df-483f-4423-b6ab-1042cbf0bae9',
 '120cbbbc-347a-402b-86b6-92f91ae4578f',
 '101da0c2-e047-45a6-98da-0777ca9427ff',
 '3b919a09-b01c-4fb1-8f87-38d50c28c70a',
 'd6f77bef-3326-4776-aa1a-de1d90c43445',
 'b30ff329-548b-

In [16]:
import os
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.graph import MessagesState, StateGraph
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_milvus import Milvus
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_core.documents import Document

load_dotenv(override=True)

def connect_milvus(embedding_model, milvus_db, milvus_host, milvus_port, milvus_collection):
    embedding_model = OpenAIEmbeddings(model=embedding_model)

    vector_store = Milvus(
        embedding_function=embedding_model,
        connection_args={
            "uri": f"http://{milvus_host}:{milvus_port}",
            "db_name": milvus_db
        },
        collection_name=milvus_collection,
    )

    return vector_store

milvus_db = os.getenv("MILVUS_DB")
milvus_host = os.getenv("MILVUS_HOST")
milvus_port = os.getenv("MILVUS_PORT")
milvus_collection = os.getenv("MILVUS_COLLECTION")

vector_store = connect_milvus("text-embedding-3-small", milvus_db, milvus_host, milvus_port, milvus_collection)
llm = init_chat_model("gpt-4.1-mini", model_provider="openai")
@tool(response_format="content_and_artifact")
def retrieve_similar_product(query: str):
    """Retrieve similar product based on user query."""
    retrieved_docs = vector_store.similarity_search(query, k=5)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n" f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

def query_or_respond_similar_product(state: MessagesState):
    """Generate tool call for retrieval or respond."""
    llm_with_tools = llm.bind_tools([retrieve_similar_product])
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tools_similar_product = ToolNode([retrieve_similar_product], name="tools_similar_product")

def generate_similar_product(state: MessagesState):
    """Generate answer."""
    recent_tool_messages = []
    for message in reversed(state["messages"]):
        print(message)
        if message.type == "tool":
            recent_tool_messages.append(message)
        else:
            break
    tool_messages = recent_tool_messages[::-1]

    docs_content = "\n\n".join(
        doc.page_content
        for m in tool_messages
        if hasattr(m, "artifact") and isinstance(m.artifact, list)
        for doc in m.artifact
        if isinstance(doc, Document)
    )

    system_message_content = (
        "You are a Customer Service Officer (CSO) assigned to recommend similar products based on a given product."
        "You have access to detailed information for each product, including its description, indication, usage, dosage, and side effects."
        "If any of these fields are missing (e.g., contain NaN), reduce the confidence score of the recommendation."
        "If no sufficiently similar product is found, respond with 'Maaf, saya belum bisa menemukan produk serupa untuk saat ini.'"
        "Your response must be concise, and for each recommended product, provide a brief description for each of them."
        "Use a natural and polite tone in your response."
        "\n\n"
        f"{docs_content}"
    )
    
    conversation_messages = [
        message
        for message in state["messages"]
        if message.type in ("human", "system")
        or (message.type == "ai" and not message.tool_calls)
    ]
    prompt = [SystemMessage(system_message_content)] + conversation_messages

    response = llm.invoke(prompt)

    return {"messages": [response]}


graph = (
    StateGraph(MessagesState)
    .add_node(query_or_respond_similar_product)
    .add_node(tools_similar_product)
    .add_node(generate_similar_product)
    .set_entry_point("query_or_respond_similar_product")
    .add_conditional_edges(
    "query_or_respond_similar_product",
        tools_condition,
        {END: END, "tools": "tools_similar_product"},
    )
    .add_edge("tools_similar_product", "generate_similar_product")
    .add_edge("generate_similar_product", END)
    .compile(name="MILVUS")
)

================================ Human Message =================================

Saya sakit maag, minum apa ya?
================================== Ai Message ==================================

Untuk meredakan sakit maag, kamu bisa minum beberapa jenis obat dan minuman yang umum direkomendasikan:

1. Antasida: Obat ini dapat menetralkan asam lambung dan memberikan efek cepat meredakan nyeri maag. Contohnya seperti Mylanta, Rolaids, atau Tums.
2. Obat penghambat asam lambung: Misalnya omeprazole atau ranitidin, yang mengurangi produksi asam lambung.
3. Minuman yang menenangkan lambung: Air putih hangat, teh herbal seperti chamomile atau jahe, dan madu bisa membantu meredakan maag.
4. Hindari konsumsi minuman berkafein, beralkohol, dan minuman bersoda karena dapat memperburuk maag.

Namun, jika sakit maag kamu sering kambuh atau parah, sebaiknya konsultasikan ke dokter untuk pemeriksaan dan pengobatan yang tepat.

Apakah kamu ingin saya carikan produk obat maag yang bisa dibeli?


In [34]:
import streamlit_authenticator as stauth
import streamlit as st
import yaml
from yaml.loader import SafeLoader
import bcrypt

password = "test123"
hashed = bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()

In [35]:
hashed

'$2b$12$5/hLuib2MLsXy4WWj2XOD.ruqSUVuRE67z9nEo2qaDbTfwfU0dtmG'

In [25]:
import streamlit_authenticator as stauth
hashed_passwords = stauth.Hasher.hash_passwords(["test123"])

TypeError: list indices must be integers or slices, not str

In [ ]:
hashed_passwords 

In [ ]:
import os
import json
import requests
from langchain_core.tools import tool

@tool("query_product_elasticsearch", parse_docstring=True)
def query_product_elasticsearch(product_name: str) -> list:
    """
    Use this tool to get the product code of similar products using Elasticsearch.

    Args:
        product_name: Name of the product provided by the customer.

    Returns:
        A list of similar product hits from Elasticsearch.
    """
    url = os.environ["ELASTIC_URL"] + "/_search"
    auth = (os.environ["ELASTIC_USERNAME"], os.environ["ELASTIC_PASSWORD"])

    query = f"""{{
        "query": {{
            "dis_max": {{
                "queries": [
                    {{"match": {{"name": {{"query": "{product_name}"}}}}}},
                    {{"match": {{"name_autocomplete": {{"query": "{product_name}"}}}}}},
                    {{"match": {{"name_synonym": {{"query": "{product_name}"}}}}}}
                ]
            }}
        }}
    }}
    """
    query = json.loads(query)
    try:
        response = requests.get(url, auth=auth, json=query)
        response.raise_for_status()
        data = response.json()
        return data.get('hits', {}).get('hits', [])
    except requests.RequestException as e:
        return [{"error": str(e)}]

In [ ]:
import os
import json
import requests
from langchain_core.tools import tool

@tool("query_product_elasticsearch", parse_docstring=True)
def query_product_elasticsearch(product_name: str) -> Dict:
    """
    Search for similar product names using Elasticsearch and return the top match and score.

    Args:
        product_name: Name or partial name of the product provided by the customer.

    Returns:
        {
            "best_match": {...},     # Top ES hit (_source + score + similarity)
            "score": float           # Jaccard similarity score
        }
    """
    url = os.environ["ELASTIC_URL"].rstrip("/") + "/_search"
    auth = (os.environ["ELASTIC_USERNAME"], os.environ["ELASTIC_PASSWORD"])

    query_elastic = {
        "query": {
            "dis_max": {
                "queries": [
                    {"match": {"name": {"query": product_name}}},
                    {"match": {"name_autocomplete": {"query": product_name}}},
                    {"match": {"name_synonym": {"query": product_name}}}
                ]
            }
        }
    }

    response = requests.get(url, auth=auth, json=query_elastic)
    response.raise_for_status()
    hits = json.dumps(response.json().get("hits", {}).get("hits", []))

In [ ]:
import cx_Oracle
import os

def oracle_set_connection(project_name, db_config):
    try:
        dsn = cx_Oracle.makedsn(db_config['HOST'], db_config['PORT'], service_name=db_config['SERVICE_NAME'])
        conn = cx_Oracle.connect(db_config['USERNAME'], db_config['PASSWORD'], dsn)
        return conn
    except Exception as e:
        print(f'Exception caught when initializing Oracle engine in {project_name}: {e}')
    return None

DB_CONFIG_ORACLE = {
    "HOST": os.environ["ORACLE_HOST"],
    "PORT": os.environ["ORACLE_PORT"],
    "SERVICE_NAME": os.environ["ORACLE_SERVICE_NAME"],
    "USERNAME": os.environ["ORACLE_USERNAME"],
    "PASSWORD": os.environ["ORACLE_PASSWORD"]
}

TABLES_ALLOWED_ORACLE = [
    { "owner": "AAM_DWH", "table": "DIM_CUSTOMER"},
    { "owner": "AAM_DWH", "table": "DIM_PRODUCT"},
    { "owner": "MISDSAAM", "table": "AAM_PRODUCT_SUGGEST_CSO"}
]

In [ ]:
@tool("get_table_list", parse_docstring=True)
def get_table_list(db_config: dict):
    """ 
        Use this tool to fetch the allowed tables from the database. The tool requires database connection and returns list of tables inside the database. 

        Args:
        db_config = database connection 

        Return: 
        list of table names
    """
    
    conn = oracle_set_connection("AI_TELECSO", db_config)
    cursor = conn.cursor()

    tables = [f"'{table['table']}'" for table in TABLES_ALLOWED_ORACLE]
    tables = ", ".join(tables)

    print(f"SELECT DISTINCT TABLE_NAME FROM ALL_TAB_COLUMNS WHERE TABLE_NAME IN ({tables})")
    cursor.execute(f"SELECT DISTINCT TABLE_NAME FROM ALL_TAB_COLUMNS WHERE TABLE_NAME IN ({tables})")
    tables_fetched = cursor.fetchall()
    return [table[0] for table in tables_fetched]

@tool("get_table_schema", parse_docstring=True)
def get_table_schema(table_list: list[str], db_config: dict):
    """
    Use this tool to fetch the table schema for allowed tables only. It returns column names, types, default values, etc.

    Args:
        table_list: list of table names
        db_config: database connection configuration

    Returns:
        A string containing schema for allowed tables only. Disabled tables are silently ignored.
    """
    table_list = [table.upper() for table in table_list]
    valid_tables = [table for table in TABLES_ALLOWED_ORACLE if table["table"] in table_list]

    if not valid_tables or len(valid_tables) == 0:
        return "No valid tables to show"
    
    conn = oracle_set_connection("AI_TELECSO", db_config)
    cursor = conn.cursor()

    output = ""

    for table in valid_tables:
        cursor.execute(f"""
            SELECT
                TABLE_NAME,
                COLUMN_NAME,
                DATA_LENGTH,
                DATA_TYPE,
                DATA_DEFAULT
            FROM
                ALL_TAB_COLUMNS
            WHERE
                OWNER = :owner
                AND TABLE_NAME = :table_name
        """, {"owner": table["owner"], "table_name": table["table"]})
        rows = cursor.fetchall()
        if not rows:
            continue

        output += f"\n📦 **{table['owner']}.{table['table']}**\n"
        output += "Column Name | Data Type | Length | Nullable | Default\n"
        output += "-"*60 + "\n"

        for col in rows:
            name, dtype, length, nullable, default = col
            output += f"{name} | {dtype} | {length} | {nullable} | {default}\n"

    conn.close()

    return output.strip() if output else "No schema info found for valid tables."

In [ ]:
get_table_schema({"table_list": get_table_list.invoke({"db_config": DB_CONFIG_ORACLE}), "db_config": DB_CONFIG_ORACLE})

In [ ]:
# Initiating Langchain Chat Models
from langchain.chat_models import init_chat_model
model = init_chat_model("gpt-4.1-mini", model_provider= "openai")

In [ ]:
from langgraph.prebuilt import ToolNode
from langgraph.graph import MessagesState
from typing import Any, Annotated, Literal, Optional, List
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage, SystemMessage

class DBGraphState(MessagesState):
    db_config: Annotated[Any, "Database connection"]
    user_question: Annotated[str, "User question that must be answered by querying the database"]
    available_tables: Optional[List[str]]

def list_tables(state: DBGraphState):
    tool_call = {
        "name": "get_table_list",
        "args": {
            "db_config": state["db_config"]
        },
        "id": "table_list",
        "type": "tool_call"
    }
    tool_call_message = AIMessage(content="I am calling a tool to get list of allowed tables from the database.", tool_calls=[tool_call])
    tool_message = get_table_list.invoke(tool_call)
    response = AIMessage(content=f"Available tables: {tool_message.content}")

    return {'messages': response, 'available_tables': tool_message.content}

def get_schema_node(state: DBGraphState):
    
    input_question = state["user_question"]
    available_tables = state["messages"][-1]
    db_config = state["db_config"]
    tool_call = {
        "name": "get_table_schema",
        "args": {
            "table_list": state["available_tables"],
            "db_config": state["db_config"]
        },
        "id": "table_list",
        "type": "tool_call"
    }
    # instruction = [SystemMessage(content=f'''You are a business analyst from Dexa and an SQL expert. You receive a question from the user and a list of available
    #                             table in the database. Use the tool to get the structures of possible tables that you will use to construct the query later. Customer and product
    #                             details are located in AAM_DWH schema, while the customer's version product details are located in MISDSAAM schema.
    #                             db_config = {json.dumps(db_config)}
    #                             Here is the question from the user: {input_question}''')
    #                 ] + [available_tables]
    # model_with_tools = model.bind_tools([get_table_schema], tool_choice="any")
    # response = model_with_tools.invoke(instruction)


    # invoking tool 
    # return {"messages": response}
    tool_message = get_table_schema.invoke(tool_call)
    response = AIMessage(content=f"Available tables: {tool_message.content}")

    return {'messages': response}

def write_query(state:DBGraphState):
    dialect = "Oracle"
    top_k = 10
    instruction = SystemMessage(content=f'''You are an agent designed to interact with a SQL database.
                        Given an input question, create a syntactically correct {dialect} query to run,
                        then look at the results of the query and return the answer. Unless the user
                        specifies a specific number of examples they wish to obtain, always limit your
                        query to at most {top_k} results.

                        Use the correct schema: 
                        - Customer and product data are in the `AAM_DWH` schema
                        - Versioned product data for each customer group is in the `MISDSAAM` schema, with all column names ending in `_SEARCH`

                        You can order the results by a relevant column to return the most interesting
                        examples in the database. Never query for all the columns from a specific table,
                        only ask for the relevant columns given the question.

                        DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.''')
    
    response = model.invoke([instruction] + state["messages"])    
   
    return {"messages": response}


invoking_tool_node = ToolNode([get_table_schema], name="invoking_tool_node")

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

builder = StateGraph(DBGraphState)
builder.add_node("get_table_list", list_tables)
builder.add_node(get_schema_node)
builder.add_node(invoking_tool_node, "invoking_tool_node")
builder.add_node(write_query)

builder.add_edge(START, "get_table_list")
builder.add_edge("get_table_list","get_schema_node")
builder.add_edge("get_schema_node","invoking_tool_node")
builder.add_edge("invoking_tool_node", "write_query")
builder.add_edge("write_query", END)

graph = builder.compile()

# Show graph 
from IPython.display import Image, display
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
user_question = "ada tabel apa saja yang bisa saya akses?"

# Apa kode produk untuk F230904176 dari relasi mitra plumbon?
response = graph.invoke({"messages":[HumanMessage(content=json.dumps({
    "user_question": user_question,
    "db_config": DB_CONFIG_ORACLE
}))], "db_config": DB_CONFIG_ORACLE, "user_question" : user_question})

for m in response['messages']: 
    m.pretty_print()

# Final Agents

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.tools import tool
from langgraph.graph import MessagesState, StateGraph
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import END
from langgraph.prebuilt import ToolNode, tools_condition

import os
import json
import requests
from langchain_core.tools import tool

from typing import Dict
import requests
import os
from langchain_core.tools import tool

llm = init_chat_model("gpt-4.1-mini", model_provider="openai")

@tool("query_product_elasticsearch", parse_docstring=True)
def query_product_elasticsearch(product_name: str):
    """
    Search for similar product names using Elasticsearch and return the top match and score.

    Args:
        product_name: Name or partial name of the product provided by the customer.

    Returns:
        Array of products with _index, _type, _id, _score, and _source. Possible to return
        empty array.
    """
    url = os.environ["ELASTIC_URL"].rstrip("/") + "/_search"
    auth = (os.environ["ELASTIC_USERNAME"], os.environ["ELASTIC_PASSWORD"])

    query_elastic = {
        "query": {
            "dis_max": {
                "queries": [
                    {"match": {"name": {"query": product_name}}},
                    {"match": {"name_autocomplete": {"query": product_name}}},
                    {"match": {"name_synonym": {"query": product_name}}}
                ]
            }
        }
    }

    response = requests.get(url, auth=auth, json=query_elastic)
    response.raise_for_status()
    hits = json.dumps(response.json().get("hits", {}).get("hits", []))
    return hits

def query_or_respond_elastic(state: MessagesState):
    """Generate tool call for retrieval or respond."""
    llm_with_tools = llm.bind_tools([query_product_elasticsearch])
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

tools_elastic = ToolNode([query_product_elasticsearch], name="tools_elastic")

def generate_elastic_code(state: MessagesState):
    """Generate answer."""
    recent_tool_messages = []
    for message in reversed(state["messages"]):
        if message.type == "tool":
            recent_tool_messages.append(message)
        else:
            break
    tool_messages = recent_tool_messages[::-1]

    docs_content = "\n\n".join(msg.content for msg in tool_messages)

    system_message_content = (
        "You are an expert Customer Service Officer (CSO) who understands how to map the customer's product name "
        "to the company's official product name. Use Elasticsearch results to determine the best match.\n\n"
    
        "Instructions:\n"
        "- If the highest `_score` in the Elasticsearch results is less than 25, you must also perform Jaccard similarity "
        "to justify your best choice.\n"
        "- Return your final answer in this format:\n"
        "  { 'product_code': _source[i].code }\n"
        "  where `i` is the index of the best-matching result.\n\n"
    
        "Here are some examples of product mappings:\n"
        "- Tride Tablet 5000 unit (vitamin D) → TRIDE 5000IU BOX 10 STR @ 6 KAP\n"
        "- Tensivask Tablet 5 MG (amlodipine) → TENSIVASK 5MG @50\n"
        "- (G) REG 3 PROPRANOLOL TABLET 10 MG DEXA → PROPRANOLOL 10MG @100(DX)\n\n"
    
        f"Elasticsearch results:\n{docs_content}\n"
    )

    conversation_messages = [
        message
        for message in state["messages"]
        if message.type in ("human", "system")
        or (message.type == "ai" and not message.tool_calls)
    ]

    prompt = [SystemMessage(system_message_content)] + conversation_messages

    response = llm.invoke(prompt)

    return {"messages": [response]}

graph = (
    StateGraph(MessagesState)
    .add_node(query_or_respond_elastic)
    .add_node(tools_elastic)
    .add_node(generate_elastic_code)
    .set_entry_point("query_or_respond_elastic")
    .add_conditional_edges(
        "query_or_respond_elastic",
        tools_condition,
        {END: END, "tools": "tools_elastic"}
    )
    .add_edge("tools_elastic", "generate_elastic_code")
    .add_edge("generate_elastic_code", END)
    .compile(name="ELASTIC")
)

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
response = graph.invoke({"messages":HumanMessage(content="Apa kode produk dari stimuno orange?")})
for m in response['messages']: 
    m.pretty_print()

In [18]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.graph import MessagesState, StateGraph
from langchain_core.messages import SystemMessage
from langgraph.prebuilt import ToolNode
from langgraph.graph import END
from typing import Any, Annotated, Literal

import os
import cx_Oracle

model = init_chat_model("gpt-4.1-mini", model_provider= "openai")

from dotenv import load_dotenv
load_dotenv(override=True)

class DBGraphState(MessagesState):
    user_question: Annotated[str, "User question that must be answered by querying the database"]
    customer_id_reference: Annotated[int, "User's customer id reference as unique identifier"]

DB_CONFIG_ORACLE = {
    "HOST": os.environ["ORACLE_HOST"],
    "PORT": os.environ["ORACLE_PORT"],
    "SERVICE_NAME": os.environ["ORACLE_SERVICE_NAME"],
    "USERNAME": os.environ["ORACLE_USERNAME"],
    "PASSWORD": os.environ["ORACLE_PASSWORD"]
}

TABLES_ALLOWED_ORACLE = [
    { "owner": "AAM_DWH", "table": "DIM_CUSTOMER"},
    { "owner": "AAM_DWH", "table": "DIM_PRODUCT"},
    { "owner": "MISDSAAM", "table": "AAM_PRODUCT_SUGGEST_CSO"}
]

def oracle_set_connection(project_name, db_config):
    try:
        dsn = cx_Oracle.makedsn(db_config['HOST'], db_config['PORT'], service_name=db_config['SERVICE_NAME'])
        conn = cx_Oracle.connect(db_config['USERNAME'], db_config['PASSWORD'], dsn)
        return conn
    except Exception as e:
        print(f'Exception caught when initializing Oracle engine in {project_name}: {e}')
    return None

@tool("query_executor", parse_docstring=True)
def query_executor(query: str):
    """
        Execute select queries to Oracle database with the provided database configurations.

        Args:
        query = select statement that will be executed
        
        Return:
        Stringified result(s) of the query.
    """
    conn = oracle_set_connection("DEMO-TELECSO", DB_CONFIG_ORACLE)
    cursor = conn.cursor()

    cursor.execute(query)
    query_result = cursor.fetchall()

    data_string = ""
    if len(query_result) == 0: 
        field_names = "No data is returned."
    else:
        field_names = " | ".join([column[0] for column in cursor.description])
        for record in query_result: 
            data_string += " | ".join([str(cell) for cell in record]) + "\n"

    output_string = f"""{field_names}\n{data_string}\n"""
    return output_string

tools_oracle_query = ToolNode([query_executor], name="tools_oracle_query")

def query_generator(state: DBGraphState):
    dialect = "Oracle"
    user_question = state['user_question']
    customer_id_reference = state['customer_id_reference']
    top_k = 10
    instruction = SystemMessage(content=f"""You are an expert data analyst designed to support Customer Service Officers (CSOs) in retrieving *product-related* information from a SQL database using {dialect} dialect. Your role is strictly limited to answering questions about product **price** or **stock** only. You must reject any other types of questions such as customer addresses, transaction history, or anything unrelated to price or stock.
                                Your behavior must follow the two-step format:
                                1. Create a **syntactically correct and safe SQL query** to answer the question using the input field `customer_id_reference` as a filter.
                                2. Return the plain SQL query strictly with the {dialect} dialect

                                User question: {user_question}
                                Customer id reference: {customer_id_reference}

                                ====================
                                ## RULES:
                                # ====================
                                # - ✅ Only answer questions about product **price** or **stock**.
                                # - ❌ DO NOT generate queries for unrelated questions (e.g., address, phone number, transaction, etc).
                                # - ❌ DO NOT include any DML or DDL operations (INSERT, UPDATE, DELETE, DROP, etc).
                                # - ✅ Always use the provided `customer_id_reference` to determine customer context.
                                # - ✅ Always restrict results to the most recent or relevant row using ROW_NUMBER() or `MAX(CUSTOMER_KEY)`.
                                # - ✅ Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.
                                # - ✅ Limit the number of returned rows and iterate if multiple results are expected.

                                ====================
                                ## REFERENCE SCHEMA:
                                ====================
                                - `AAM_DWH.DIM_CUSTOMER`
                                - `MISDSAAM.AAM_PRODUCT_SUGGEST_CSO`
                                - `MISDSAAM.AAM_PRODUCT_STOCK_CSO`
                                - `AAM_DWH.DIM_PRODUCT`

                                These tables are joined based on:
                                - CUSTOMER_GROUP_ID
                                - PRODUCT_CODE

                                ====================
                                ## EXAMPLE 1: Product Price
                                ====================
                                **User Question**: What is the current price of "0102060052"?  
                                **customer_id_reference**: {customer_id_reference}                        
                                **SQL Query**:
                                SELECT CUSTOMER_KEY, PRODUCT_KEY, CUSTOMER_GROUP_ID, CUSTOMER_GROUP_DESC, 
                                    PRODUCT_CODE, PRODUCT_DESC, PRODUCT_CODE_SEARCH, PRODUCT_DESC_SEARCH, HNA
                                FROM (
                                    SELECT 
                                        dc.CUSTOMER_KEY,
                                        dp.PRODUCT_KEY,
                                        dc.CUSTOMER_GROUP_ID,
                                        dc.CUSTOMER_GROUP_DESC,
                                        apsc.PRODUCT_CODE,
                                        apsc.PRODUCT_DESC,
                                        apsc.PRODUCT_CODE_SEARCH,
                                        apsc.PRODUCT_DESC_SERACH,
                                        dp.HNA,
                                        ROW_NUMBER() OVER (
                                            ORDER BY dc.CUSTOMER_KEY DESC, dp.PRODUCT_KEY DESC
                                        ) AS rn
                                    FROM AAM_DWH.DIM_CUSTOMER dc
                                    INNER JOIN MISDSAAM.AAM_PRODUCT_SUGGEST_CSO apsc 
                                        ON dc.CUSTOMER_GROUP_ID = apsc.CUSTOMER_GROUP_ID
                                    INNER JOIN AAM_DWH.DIM_PRODUCT dp 
                                        ON dp.PRODUCT_CODE = apsc.PRODUCT_CODE
                                    WHERE dc.CUSTOMER_ID_REF = {customer_id_reference}
                                AND apsc.PRODUCT_CODE_SEARCH = '0102060052'
                                )
                                WHERE rn = 1;

                                ====================
                                ## EXAMPLE 2: Product Stock
                                ====================
                                **User Question**: What is the stock of "stimuno"?  
                                **customer_id_reference**: {customer_id_reference}
                                **SQL Query**:
                                WITH max_customer AS (
                                    SELECT MAX(CUSTOMER_KEY) AS max_customer_key
                                    FROM AAM_DWH.DIM_CUSTOMER
                                    WHERE CUSTOMER_ID_REF = {customer_id_reference}
                                )
                                SELECT 
                                    dc.CUSTOMER_KEY,
                                    dp.PRODUCT_KEY,
                                    dc.CUSTOMER_GROUP_ID,
                                    dc.CUSTOMER_GROUP_DESC,
                                    apsc.PRODUCT_CODE,
                                    apsc.PRODUCT_DESC,
                                    apsc.PRODUCT_DESC_SEARCH,
                                    apsc.PRODUCT_CODE_SEARCH,
                                    apstc.STOCK
                                FROM AAM_DWH.DIM_CUSTOMER dc
                                INNER JOIN MISDSAAM.AAM_PRODUCT_SUGGEST_CSO apsc 
                                    ON dc.CUSTOMER_GROUP_ID = apsc.CUSTOMER_GROUP_ID
                                INNER JOIN AAM_DWH.DIM_PRODUCT dp 
                                    ON dp.PRODUCT_CODE = apsc.PRODUCT_CODE
                                INNER JOIN MISDSAAM.AAM_PRODUCT_STOCK_CSO apstc
                                    ON dp.PRODUCT_CODE = apstc.PRODUCT_CODE
                                JOIN max_customer mc 
                                    ON dc.CUSTOMER_KEY = mc.max_customer_key
                                WHERE dc.CUSTOMER_ID_REF = {customer_id_reference}
                                AND apsc.PRODUCT_DESC_SEARCH LIKE '%STIM%';

                                ====================
                                ## OUT OF SCOPE
                                ====================
                                If the user asks anything **outside price or stock**, answer:
                                "Maaf, saya hanya bisa membantu pertanyaan seputar harga dan stok produk."
                                ====================
                                """
                            )
    response = model.invoke([instruction] + state["messages"])
    return {"messages": response}  

def check_query(state: DBGraphState):
    dialect = 'Oracle'
    instruction = SystemMessage(content=f'''You are a SQL expert with a strong attention to detail.
    Double check the {dialect} query for common mistakes, including:
    - Using NOT IN with NULL values
    - Using UNION when UNION ALL should have been used
    - Using BETWEEN for exclusive ranges
    - Data type mismatch in predicates
    - Properly quoting identifiers
    - Using the correct number of arguments for functions
    - Casting to the correct data type
    - Using the proper columns for joins

    If there are any of the above mistakes, rewrite the query. If there are no mistakes,
    just reproduce the original query.

    Forbid any DML statements (INSERT, UPDATE, DELETE, DROP, TRUNCATE). If the query statement contains those statements, respond by "Forbidden query"
    ''')

    response = model.invoke([instruction] + state["messages"])
    if isinstance(response, AIMessage):
        content = response.content.strip()
        return { "query": content }
    else:
        print("Response is not an AI message")
        print(response)
        
def run_query_node(state: DBGraphState):
    query_checking_result = state["messages"][-1]
    dialect = 'sqlite'
    instruction = [SystemMessage(content=f'''If the last node is resulted in a forbidden query, proceed to the next node, explain why it is forbidden and skip calling tool.
                                 If the result is a valid {dialect} query statement, run the query by calling the given tool.
                                '''), query_checking_result]
    model_with_tools = model.bind_tools([query_executor])
    response = model_with_tools.invoke(instruction)

    return {"messages": response}

def final_answer(state: DBGraphState):
    user_question = state['user_question']
    customer_id_reference = state['customer_id_reference']
    query_result = state['messages'][-1]
    
    instruction = [SystemMessage(content=f'''You are an intelligent assistant tasked with answering the user's question based on query results from a SQL database.
Carefully analyze the SQL result and generate a natural language response that directly answers the user's question.

**INSTRUCTIONS**:
1. Determine if the SQL query result contains enough data to answer the user question.
2. If YES:
   - Extract and show **all rows** of relevant products.
   - Replace placeholders like `[PRODUCT_DESC_SEARCH]`, `[PRODUCT_CODE_SEARCH]`, and `[HNA]` with actual values from the query.
   - Summarize the result clearly using bullet points (one bullet per row).
   - Use a **natural and polite tone**.
3. Replace all placeholders (e.g., [PRODUCT_DESC_SEARCH], [PRODUCT_CODE_SEARCH], [HNA], etc.) with actual values from the SQL result.
4. If multiple results are returned, summarize them clearly using bullet points.
5. If the query does **NOT** provide enough information, politely ask the user for a more specific product name or detail.
   - Example: 
     ```
     Hasil tidak memuat cukup detail untuk menjawab pertanyaan Anda secara akurat.
     Mohon berikan nama produk atau kode produk yang lebih spesifik agar saya bisa membantu lebih tepat.
     ```
6. If the question is **not related to price or stock**, respond with:
  "Maaf, saya hanya bisa membantu pertanyaan seputar harga dan stok produk."

**USER QUESTION**: {user_question}
**CUSTOMER_ID_REFERENCE** {customer_id_reference}

**QUERY RESULT**:
{query_result}

====================
## REFERENCE SCHEMA:
====================
- `AAM_DWH.DIM_CUSTOMER`
- `MISDSAAM.AAM_PRODUCT_SUGGEST_CSO`
- `MISDSAAM.AAM_PRODUCT_STOCK_CSO`
- `AAM_DWH.DIM_PRODUCT`

These tables are joined based on:
- CUSTOMER_GROUP_ID
- PRODUCT_CODE

====================
## EXAMPLE 1: Product Price
====================
**User Question**: Berapa harga "0102060052"?  
**Answer**:  
Harga dari produk [PRODUCT_DESC_SEARCH] dengan kode [PRODUCT_CODE_SEARCH], yaitu [PRODUCT_DESC] adalah Rp. [HNA].

====================
## EXAMPLE 2: Product Stock
====================
**User Question**: Ada stok untuk "stimuno"?  
**Answer**:  
Stok dari produk [PRODUCT_DESC_SEARCH] dengan kode [PRODUCT_CODE_SEARCH], yaitu [PRODUCT_DESC] adalah [STOCK].

====================
## OUT OF SCOPE
====================
If the user asks anything outside price or stock:
"Maaf, saya hanya bisa membantu pertanyaan seputar harga dan stok produk."
''')]

    response = model.invoke(instruction)
    return {"messages": response}

graph = (
    StateGraph(DBGraphState)
    .add_node("query_generator", query_generator)
    .add_node("query_checker", check_query)
    .add_node("tools_oracle_query", tools_oracle_query)
    .add_node("run_query_node", run_query_node)
    .add_node("final_answer", final_answer)
    .add_edge("query_generator", "tools_oracle_query")
    .add_edge("tools_oracle_query", "query_checker")
    .add_edge("query_checker", "run_query_node")
    .add_edge("run_query_node", "final_answer")
    .add_edge("final_answer", END)
    .set_entry_point("query_generator")
    .compile()
)

In [19]:
from langchain_core.messages import HumanMessage, AIMessage
import json
response = graph.invoke({"messages":[HumanMessage(content=json.dumps({
            "user_question": "apakah stok stimuno ada?",
            "db_config": DB_CONFIG_ORACLE,
            "customer_id_reference": 126851
        }))], "db_config": DB_CONFIG_ORACLE, "user_question" : "apakah stok stimuno ada?", "customer_id_reference": 126851})

In [20]:
response['messages'][-1]

AIMessage(content='Berikut informasi stok untuk produk yang mengandung kata "stimuno" sesuai dengan data untuk customer ID 126851:\n\n- Produk STIMUNO DC dengan kode produk CS25613: stok tersedia sebanyak 3826.\n\nJadi, stok untuk produk Stimuno masih tersedia saat ini. Apabila Anda memerlukan informasi produk lainnya, silakan beritahu saya.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 1476, 'total_tokens': 1549, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': None, 'id': 'chatcmpl-BwQfOyaJnFFZFL9q0lsEaGw5ElaGG', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--ef5a24bd-b7fb-4789-92b4-46b2ac59e7a1-0', usage_metadata={'input_tokens': 1476, 'output_tokens': 73, 'total_to

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
response = graph.invoke({"messages":[HumanMessage(content="Stok stimuno sisa berapa?")], "user_question": "Stok stimuno sisa berapa?", "customer_id_reference": 126851})
for m in response['messages']:
    if isinstance(m, AIMessage):
        m.pretty_print()
        
    else:
        print(f"[{m.type}] {m.content}")